In [ ]:
# Requirements

import os
os.environ["NGSPICE_LIBRARY_PATH"] = "/opt/homebrew/lib/libngspice.dylib"

from PySpice.Unit import *
from pid_spice import PID, PIDConfig
from tune import PIDFactory, Tuner
from cache import Cache
import matplotlib.pyplot as plt

In [ ]:
# Make plots vecorized

%config InlineBackend.figure_format = 'svg'
plt.rcParams["savefig.format"] = "svg"

In [ ]:
# Cache to checkpoint already done work

cache   = Cache()

In [ ]:
factory = PIDFactory()                              # to tune
                                 # to cache tuning work

tuner = Tuner(
    factory      = factory,
    setpoint     = 1.0,
    drive_kwargs = {"kind": "step", "amplitude": 1.0, "t0": 1e-3},
)


@cache
def run_tuner():
    return tuner.run(
        weights = {
            "steady_state_error" : 5.0,
            "overshoot_%"        : 1.0,
            "settling_time_s"    : 1.0,
            "rise_time_s"        : 0.3,
        },
        max_iter = 500,
        tol      = 1e-3,
        verbose  = True,
    )


result = run_tuner()

tuner._best_params = result["best_params"]
tuner._best_stats  = result["best_stats"]
tuner._history     = result["history"]

best_cfg = tuner.best_config()

tuner.plot_best()
tuner.plot_history()

In [ ]:
cfg = PIDConfig()
# cfg = best_cfg

# cfg.reconfig({
#     "RIN_PROP": 120@u_kΩ,
#     "RFB_PROP": 120@u_kΩ,
#     "C_INT": 8@u_uF,
#     "RDAMP_DIFF": 200@u_Ω
# })


pid = PID(cfg)
pid._system()
print(pid.configure())
pid.drive(kind="step", amplitude=2.0, t0=1e-3)
pid.plot()



In [ ]:
# how to export ltspice runable file
pid.export_circuit()